In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import pymongo
import os
from tensorflow.keras.utils import plot_model
import gc
from tqdm import tqdm
import tensorflow as tf
from numba import cuda
import time

In [ ]:
def create_new_version_dir(base_path):
    version = 1
    while True:
        version_dir = os.path.join(base_path, f'v{version}')
        if not os.path.exists(version_dir):
            os.makedirs(version_dir)
            print(f"Created new directory: {version_dir}")
            return version_dir
        version += 1

In [ ]:
# Funktion zur Überwachung des Speicherverbrauchs
def check_memory_usage(threshold=90):
    memory = psutil.virtual_memory()
    if memory.percent > threshold:
        print(f"Warnung: Speicherverbrauch bei {memory.percent}%. Programm wird gestoppt.")
        raise MemoryError("Speicherverbrauch zu hoch")

In [ ]:
# Load and Merge Data (Assuming MongoDB setup is correct)
def load_data():
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    canvassamples = list(db['canvassamples'].find())
    fingerprints = list(db['fingerprints'].find())
    
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    merged_df = pd.merge(canvassamples_df, fingerprints_df, 
                         left_on='fingerprintId', 
                         right_on='_id', 
                         suffixes=('_sample', '_fingerprint'))
    
    return merged_df

merged_df = load_data()
print(f"Gesamtdatensatz enthält {len(merged_df)} Einträge.")

In [ ]:
# Preprocess Data

# Function to process images in RGB
def process_image_rgb(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')  # Convert image to RGB
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0  # Normalize to values between 0 and 1
        return image_array
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

# Function to extract images and labels from DataFrame
def extract_images_from_df(df, example_user_id):
    images = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Lade Bilder"):
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Example user ID
example_user_id = 'benutzername_1'

# Create DataFrames for each user
user_ids = merged_df['username'].unique()
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Sample data for the example user
user_df = user_dfs[example_user_id].sample(n=12000, random_state=42)

# Add negative examples and limit
negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
negative_df = negative_df.sample(n=12000, random_state=42)

# Split into Train/Val/Test sets (70% Train, 20% Val, 10% Test)
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

# Process image data
try:
    X_train, y_train = extract_images_from_df(train_df, example_user_id)
    X_val, y_val = extract_images_from_df(val_df, example_user_id)
    X_test, y_test = extract_images_from_df(test_df, example_user_id)
except Exception as e:
    print(f"Fehler bei der Bilddatenverarbeitung: {e}")

# Free up memory
gc.collect()

In [ ]:
import os
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
import numpy as np
from numba import cuda

# Function to get the latest version directory
def get_latest_version_dir(base_path):
    version = 1
    latest_version_dir = None
    while True:
        version_dir = os.path.join(base_path, f'v{version}')
        if not os.path.exists(version_dir):
            break
        latest_version_dir = version_dir
        version += 1
    return latest_version_dir

In [ ]:
# Function to prepare images from the dataframe
def prepare_images(df, num_samples, example_user_id):
    images = []
    labels = []
    
    # Ensure at least 1000 samples from the example user
    example_user_samples = df[df['username'] == example_user_id].sample(n=1000, random_state=42)
    other_samples = df[df['username'] != example_user_id].sample(n=num_samples, random_state=42)
    
    sampled_df = pd.concat([example_user_samples, other_samples])
    
    for _, row in sampled_df.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(row['username'])
    
    return np.array(images), np.array(labels)

In [ ]:
# Test each model with 1000 samples from random users
num_samples = 1000
base_path = '/ssd'  # Ensure this is the correct base path
latest_version_dir = get_latest_version_dir(base_path)
model_names = ['create_model_1', 'create_model_2', 'create_model_3', 'create_model_4', 'create_model_5']
sample_sizes = [100, 200, 500, 1000, 2000, 5000, 10000]
negative_sample_ratios = [0.5, 1, 2]

results = []
example_user_id = 'benutzername_1'  # Beispiel-Benutzer-ID
batch_size = 8  # Reduce batch size to avoid memory issues

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
from tensorflow.keras.models import load_model

def clear_gpu_memory():
    tf.keras.backend.clear_session()
    tf.compat.v1.reset_default_graph()

results = []
actual_predicted_results = []

for model_name in model_names:
    for sample_size in sample_sizes:
        for neg_ratio in negative_sample_ratios:
            print(f"Testing model {model_name} with {num_samples} samples from random users")
            
            model_path = os.path.join('/ssd/v1', f'{model_name}_samplesize_{sample_size}_negratio_{neg_ratio}.h5')
            print(f"Checking model path: {model_path}")
            if not os.path.exists(model_path):
                print(f"Error loading model: No file or directory found at {model_path}")
                continue
            
            try:
                model = load_model(model_path)
                print(f"Loaded model from: {model_path}")
            except Exception as e:
                print(f"Error loading model: {e}")
                continue
            
            X_test_users, y_test_users = prepare_images(merged_df, num_samples, example_user_id)
            print(f"Test data shape: {X_test_users.shape}, Labels shape: {y_test_users.shape}")
            
            y_pred = []
            current_batch_size = batch_size
            
            while True:
                try:
                    with tf.device('/GPU:0'):
                        for i in range(0, len(X_test_users), current_batch_size):
                            batch = X_test_users[i:i + current_batch_size]
                            batch = tf.convert_to_tensor(batch)
                            y_pred_batch = model.predict(batch)
                            y_pred.extend(y_pred_batch)
                    break
                except tf.errors.ResourceExhaustedError:
                    print(f"GPU memory exhausted with batch size {current_batch_size}, reducing batch size")
                    clear_gpu_memory()
                    current_batch_size = max(1, current_batch_size // 2)
                    if current_batch_size == 1:
                        print("Switching to CPU due to insufficient GPU memory")
                        with tf.device('/CPU:0'):
                            for i in range(0, len(X_test_users), current_batch_size):
                                batch = X_test_users[i:i + current_batch_size]
                                batch = tf.convert_to_tensor(batch)
                                y_pred_batch = model.predict(batch)
                                y_pred.extend(y_pred_batch)
                        break
            
            y_pred_labels = [example_user_id if pred > 0.5 else 'other_user' for pred in np.array(y_pred).flatten()]
            
            results_df = pd.DataFrame({
                'Actual_User': y_test_users,
                'Predicted_User': y_pred_labels
            })
            
            accuracy = accuracy_score(results_df['Actual_User'] == example_user_id, results_df['Predicted_User'] == example_user_id)
            precision = precision_score(results_df['Actual_User'], results_df['Predicted_User'], average='macro')
            recall = recall_score(results_df['Actual_User'], results_df['Predicted_User'], average='macro')
            f1 = f1_score(results_df['Actual_User'], results_df['Predicted_User'], average='macro')
            fpr, tpr, _ = roc_curve(results_df['Actual_User'] == example_user_id, results_df['Predicted_User'] == example_user_id)
            roc_auc = auc(fpr, tpr)
            
            print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, ROC AUC: {roc_auc:.4f}")
            
            results.append((model_name, sample_size, neg_ratio, accuracy, precision, recall, f1, roc_auc))
            
            # Save actual and predicted results for later use
            for actual, predicted in zip(y_test_users, y_pred_labels):
                actual_predicted_results.append((model_name, actual, predicted))

# Convert results to DataFrame
results_df = pd.DataFrame(results, columns=['Model', 'Sample Size', 'Negative Ratio', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'])

# Save the results to a CSV file
results_df.to_csv('model_evaluation_results.csv', index=False)

# Save the actual and predicted results to a CSV file
actual_predicted_df = pd.DataFrame(actual_predicted_results, columns=['Model', 'Actual_User', 'Predicted_User'])
actual_predicted_df.to_csv('actual_predicted_results.csv', index=False)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the results from the CSV file
results_df = pd.read_csv('model_evaluation_results.csv')

# Plotting the results for each metric
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
for metric in metrics:
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Model', y=metric, hue='Negative Ratio', data=results_df)
    plt.title(f'Model {metric} Comparison')
    plt.xticks(rotation=45)
    plt.legend(title='Negative Ratio')
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the results from the CSV file
results_df = pd.read_csv('model_evaluation_results.csv')

# Check the columns in the DataFrame
print(results_df.columns)

# Plotting the results for each metric
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
for metric in metrics:
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Model', y=metric, hue='Negative Ratio', data=results_df)
    plt.title(f'Model {metric} Comparison')
    plt.xticks(rotation=45)
    plt.legend(title='Negative Ratio')
    plt.show()

# Display the results DataFrame
print(results_df)

# Determine the best performing model for each metric
best_models = {}
for metric in metrics:
    best_model = results_df.loc[results_df[metric].idxmax()]
    best_models[metric] = best_model
    print(f"Best Model for {metric}:")
    print(best_model)
    print()

# Determine the best performing model based on the average rank of all metrics
results_df['Average Rank'] = results_df[metrics].rank(ascending=False).mean(axis=1)
best_overall_model = results_df.loc[results_df['Average Rank'].idxmin()]

print("Best Overall Performing Model:")
print(best_overall_model)

# Plot the performance of the best overall model
plt.figure(figsize=(12, 8))
sns.barplot(x=metrics, y=best_overall_model[metrics])
plt.title(f'Performance of the Best Overall Model: {best_overall_model["Model"]}')
plt.show()

# Summary of best models for each metric
summary_df = pd.DataFrame(best_models).T
summary_df = summary_df[['Model', 'Sample Size', 'Negative Ratio', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']]
print("Summary of Best Models for Each Metric:")
print(summary_df)

# Save the summary to a CSV file
summary_df.to_csv('best_models_summary.csv', index=False)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

# Load the results from the CSV file
results_df = pd.read_csv('model_evaluation_results.csv')

# Check the columns in the DataFrame
print(results_df.columns)

# Plotting the F1 Score and F1 Matrix for each model
for model_name in results_df['Model'].unique():
    model_results = results_df[results_df['Model'] == model_name]
    
    # Ensure the correct column names are used
    if 'F1 Score' in model_results.columns:
        f1_scores = model_results['F1 Score']
        
        # Plot F1 Score
        plt.figure(figsize=(12, 8))
        sns.barplot(x='Sample Size', y='F1 Score', hue='Negative Ratio', data=model_results)
        plt.title(f'F1 Score for {model_name}')
        plt.xlabel('Sample Size')
        plt.ylabel('F1 Score')
        plt.legend(title='Negative Ratio')
        plt.show()
        
        # F1 Matrix
        # Assuming 'Actual_User' and 'Predicted_User' are in a different DataFrame
        # Load the actual and predicted user data
        actual_predicted_df = pd.read_csv('actual_predicted_results.csv')
        actual_predicted_results = actual_predicted_df[actual_predicted_df['Model'] == model_name]
        
        if 'Actual_User' in actual_predicted_results.columns and 'Predicted_User' in actual_predicted_results.columns:
            y_true = actual_predicted_results['Actual_User']
            y_pred = actual_predicted_results['Predicted_User']
            
            report = classification_report(y_true, y_pred, output_dict=True)
            f1_matrix = pd.DataFrame(report).transpose()
            
            plt.figure(figsize=(10, 7))
            sns.heatmap(f1_matrix[['f1-score']].iloc[:-1, :], annot=True, cmap='Blues')
            plt.title(f'F1 Score Matrix for {model_name}')
            plt.xlabel('Metrics')
            plt.ylabel('Classes')
            plt.show()
        else:
            print(f"Columns 'Actual_User' and 'Predicted_User' not found in the results for model {model_name}")
    else:
        print(f"Column 'F1 Score' not found in the results for model {model_name}")